In [1]:
import pandas as pd
import ast

df = pd.read_csv("movie_genre_metadata.csv")
print(df.shape)
df.head()

(4000, 17)


,id,title,genres,overview,popularity,release_date,budget,revenue,tagline,original_language,runtime,image_width,image_height,caption,caption_with_overview,genre_names,num_genres
0,937249,57 Seconds,"[{'id': 53, 'name': 'Thriller'} {'id': 878, 'n...",When a tech blogger lands an interview with a ...,1036.051,2023-09-29,0,0,Rewind the past. Avenge the future.,en,99,500,750,"Title: 57 Seconds\nGenres: Thriller, Science F...","Title: 57 Seconds\nGenres: Thriller, Science F...","['Thriller', 'Science Fiction', 'Action']",3
1,575264,Mission: Impossible - Dead Reckoning Part One,"[{'id': 28, 'name': 'Action'} {'id': 53, 'name...",Ethan Hunt and his IMF team embark on their mo...,3132.329,2023-07-08,291000000,567148955,We all share the same fate.,en,164,500,750,Title: Mission: Impossible - Dead Reckoning Pa...,Title: Mission: Impossible - Dead Reckoning Pa...,"['Action', 'Thriller']",2
2,299054,Expend4bles,"[{'id': 28, 'name': 'Action'} {'id': 12, 'name...",Armed with every weapon they can get their han...,1063.874,2023-09-15,100000000,30000000,They'll die when they're dead.,en,103,500,750,"Title: Expend4bles\nGenres: Action, Adventure,...","Title: Expend4bles\nGenres: Action, Adventure,...","['Action', 'Adventure', 'Thriller']",3
3,678512,Sound of Freedom,"[{'id': 28, 'name': 'Action'} {'id': 18, 'name...","The story of Tim Ballard, a former US governme...",1271.779,2023-07-03,15000000,238000000,Fight for the light. Silence the darkness.,en,131,500,750,"Title: Sound of Freedom\nGenres: Action, Drama","Title: Sound of Freedom\nGenres: Action, Drama...","['Action', 'Drama']",2
4,1151534,Nowhere,"[{'id': 53, 'name': 'Thriller'} {'id': 18, 'na...",A young pregnant woman named Mia escapes from ...,1290.251,2023-09-29,0,0,Attempting to survive in the middle of nowhere...,es,109,500,750,"Title: Nowhere\nGenres: Thriller, Drama","Title: Nowhere\nGenres: Thriller, Drama\nOverv...","['Thriller', 'Drama']",2


In [2]:
df["genre_names"] = df["genre_names"].apply(ast.literal_eval)
print(type(df["genre_names"].iloc[0]))
print(df["genre_names"].iloc[0])

<class 'list'>
['Thriller', 'Science Fiction', 'Action']


In [3]:
print("Prije čišćenja:", len(df))

df = df[df["genre_names"].apply(len) > 0].reset_index(drop=True)

print("Nakon uklanjanja filmova bez žanra:", len(df))

Prije čišćenja: 4000
Nakon uklanjanja filmova bez žanra: 3985


In [4]:
print(df.isnull().sum())
print()
print("Broj filmova s budget=0:", (df["budget"] == 0).sum())
print("Broj filmova s revenue=0:", (df["revenue"] == 0).sum())
print("Broj filmova s runtime=0:", (df["runtime"] == 0).sum())

id                          0
title                       0
genres                      0
overview                   19
popularity                  0
release_date                0
budget                      0
revenue                     0
tagline                  1008
original_language           0
runtime                     0
image_width                 0
image_height                0
caption                     0
caption_with_overview       0
genre_names                 0
num_genres                  0
dtype: int64

Broj filmova s budget=0: 1862
Broj filmova s revenue=0: 1690
Broj filmova s runtime=0: 41


In [5]:
df["budget_known"] = df["budget"] > 0
df["revenue_known"] = df["revenue"] > 0

print(df["budget_known"].value_counts())
print(df["revenue_known"].value_counts())

True     2123
False    1862
Name: budget_known, dtype: int64
True     2295
False    1690
Name: revenue_known, dtype: int64


In [6]:
median_runtime = df[df["runtime"] > 0]["runtime"].median()
print("Median runtime:", median_runtime)

df.loc[df["runtime"] == 0, "runtime"] = median_runtime

print("Broj filmova s runtime=0 nakon imputacije:", (df["runtime"] == 0).sum())

Median runtime: 103.0
Broj filmova s runtime=0 nakon imputacije: 0


In [7]:
df["overview"] = df["overview"].fillna("")
df["tagline"] = df["tagline"].fillna("")

print(df["overview"].isnull().sum(), df["tagline"].isnull().sum())

0 0


In [8]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df["genre_names"])

print("Broj žanrova (klasa):", len(mlb.classes_))
print("Klase:", mlb.classes_)
print("Shape matrice labela:", genre_matrix.shape)

Broj žanrova (klasa): 19
Klase: ['Action' 'Adventure' 'Animation' 'Comedy' 'Crime' 'Documentary' 'Drama'
 'Family' 'Fantasy' 'History' 'Horror' 'Music' 'Mystery' 'Romance'
 'Science Fiction' 'TV Movie' 'Thriller' 'War' 'Western']
Shape matrice labela: (3985, 19)


In [9]:
genre_counts = pd.Series(genre_matrix.sum(axis=0), index=mlb.classes_).sort_values(ascending=False)
print(genre_counts)

Drama              1459
Action             1181
Thriller           1123
Comedy             1037
Adventure           803
Horror              660
Fantasy             583
Animation           567
Science Fiction     548
Romance             536
Crime               516
Family              508
Mystery             384
History             176
War                 122
Music                94
Documentary          67
TV Movie             55
Western              40
dtype: int64


In [10]:
print("Najčešći žanr:", genre_counts.idxmax(), "-", genre_counts.max(), "primjera")
print("Najrjeđi žanr:", genre_counts.idxmin(), "-", genre_counts.min(), "primjera")
print("Omjer neuravnoteženosti:", round(genre_counts.max() / genre_counts.min(), 1))

Najčešći žanr: Drama - 1459 primjera
Najrjeđi žanr: Western - 40 primjera
Omjer neuravnoteženosti: 36.5


In [14]:
from sklearn.model_selection import train_test_split
import numpy as np

strat_col = df["num_genres"].apply(lambda x: str(x) if x <= 5 else "5+")
print(strat_col.value_counts())

indices = np.arange(len(df))

train_idx, temp_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=strat_col
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, random_state=42, stratify=strat_col.iloc[temp_idx]
)

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

3     1449
2     1151
1      654
4      542
5      152
5+      37
Name: num_genres, dtype: int64
Train: 3188, Val: 398, Test: 399
